# Reading at lower resolution

What does `load_digital_number(resolution="1000m")` actually give you?
Let's open the test product at 1 km and check a few cells by hand.

In [ ]:
import os
import warnings

import numpy as np
import rasterio
from rasterio.windows import Window

from safe_s1 import Sentinel1Reader, sentinel1_xml_mappings
from safe_s1.reader import _float_vrt

warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

filename = sentinel1_xml_mappings.get_test_file(
    "S1A_IW_GRDH_1SDV_20170907T103020_20170907T103045_018268_01EB76_Z010.SAFE"
)
reader = Sentinel1Reader(name=filename)
res, ds = reader.load_digital_number(
    resolution="1000m",
    chunks={"line": 5000, "sample": 5000},
)
ds

## Cells of 100 x 100 pixels

The pixels are 10 m, so a 1 km cell is 100 x 100 pixels.
The image is 16 778 lines long: that makes 167 whole cells, and the last 78 lines are left out.

Cell `k` covers pixels `100 k` to `100 k + 99`, and its coordinate is its center:
49.5, 149.5, 249.5...

In [ ]:
ds.line.values[:3], ds.line.values[-1]

## One cell, by hand

Let's take cell (83, 125), in the middle of the scene, read its 100 x 100 pixels
in the measurement file, and compute their rms (sigma0 goes with DN², so the rms of the DN
is the DN of the mean intensity). Same number:

In [ ]:
vv_file = os.path.join(reader.path, reader.files["measurement"].iloc[0])

with rasterio.open(vv_file) as src:
    pixels = src.read(1, window=Window(125 * 100, 83 * 100, 100, 100)).astype(float)

print("by hand:", np.sqrt(np.mean(pixels**2)).round(2))
print("read:   ", ds.digital_number.sel(pol="VV").values[83, 125].round(2))

## The edge of the swath

Outside the swath, the file holds zeros. This product starts with 5 lines and 5 columns of them:

In [ ]:
with rasterio.open(vv_file) as src:
    corner = src.read(1, window=Window(0, 0, 100, 100)).astype(float)

corner[:8, :8].astype(int)

These zeros are not signal, so they are left out of the mean: the corner cell is the rms of its
95 x 95 real pixels. `valid_fraction` tells how much of the cell is inside the swath,
here 95 x 95 / 100 x 100 = 0.9025. Handy to keep only full cells: `ds.where(ds.valid_fraction == 1)`.

In [ ]:
valid = corner > 0
print("by hand:       ", np.sqrt(np.mean(corner[valid] ** 2)).round(2))
print("read:          ", ds.digital_number.sel(pol="VV").values[0, 0].round(2))
print("valid_fraction:", ds.valid_fraction.values[0, 0])

## How it's done

The file is uint16. Under the hood, safe_s1 hands GDAL a tiny VRT that says
"read this file as float32, and 0 means no data":

In [ ]:
print(_float_vrt(vv_file, nodata=0))

GDAL then averages each cell itself, in float32 (no rounding to integers) and skipping the zeros.
The read window stops at the last whole cell, so every cell is exactly 100 x 100 pixels.

`valid_fraction` comes from two averages of the same cell: one counting the zeros, one without.
Their ratio is the share of valid pixels.

For non-integer cells (an EW product at 40 m read at `"100m"` gives 2.5 pixels),
a pixel cut by the cell edge counts for the part of it inside the cell.
All the details are in [load_digital_number](../api.rst#safe_s1.Sentinel1Reader.load_digital_number).